Building an End-to-End Data Engineering Pipeline for E-Commerce Order Analytics

Step 1: EXTRACT - Baca Data Mentah

In [31]:
from pathlib import Path
import pandas as pd
import numpy as np  
import ast
# Import semua data raw 
data = pd.read_csv("../data/raw/raw_promo.csv")

data.head(10)

,promo_code,nama_promo,tipe_diskon,nilai_diskon,tanggal_mulai,tanggal_selesai,status
0,KAHFBARU,Diskon Pengguna Baru,fixed,15000.0,2024-01-01,2024-12-31,active
1,GAJIAN50,Diskon Gajian,fixed,20000.0,2024-04-25,2024-05-05,active
2,RAMADHAN,promo ramadhan,percentage,15.0,2024-03-10,2024-04-10,expired
3,FACECARE10,Diskon Kategori Face Care,percentage,10.0,2024-05-01,2024-07-31,Active
4,FLASH25,Flash Sale 25%,percentage,25.0,2024-06-01,2024-06-03,expired
5,PARFUMKAHF,Diskon Fragrance,percentage,12.0,2024-05-01,2024-08-31,active
6,GRATISONGKIR,Gratis Ongkir,fixed,10000.0,2024-02-01,2024-02-29,expired
7,VIP20,Diskon Member VIP,percentage,20.0,2024-01-15,2024-12-31,active
8,WEEKEND15,diskon akhir pekan,percentage,NaN,2024-06-01,2024-08-31,active
9,CUCI GUDANG,Cuci Gudang Juli,fixed,15000.0,2024-07-01,2024-07-15,Expired


In [32]:
# Inspeksi awal
print(f"Jumlah baris: {len(data)}")
print(f"Kolom: {list(data.columns)}")
data.info()

Jumlah baris: 10
Kolom: ['promo_code', 'nama_promo', 'tipe_diskon', 'nilai_diskon', 'tanggal_mulai', 'tanggal_selesai', 'status']
<class 'pandas.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 7 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   promo_code       10 non-null     str    
 1   nama_promo       10 non-null     str    
 2   tipe_diskon      10 non-null     str    
 3   nilai_diskon     9 non-null      float64
 4   tanggal_mulai    10 non-null     str    
 5   tanggal_selesai  10 non-null     str    
 6   status           10 non-null     str    
dtypes: float64(1), str(6)
memory usage: 692.0 bytes


In [33]:
print("\nDuplikasi")
print(f"{data.duplicated().sum()}")


Duplikasi
0


In [34]:
print("\nMissing values:")
print(data.isnull().sum())


Missing values:
promo_code         0
nama_promo         0
tipe_diskon        0
nilai_diskon       1
tanggal_mulai      0
tanggal_selesai    0
status             0
dtype: int64


Step 2: TRANSFORM - Bersihkan Data

In [35]:
# Nama promo: Huruf awal setiap kata menjadi kapital
data['nama_promo'] = data['nama_promo'].str.strip().str.title()

# Tipe diskon: Semua huruf kecil
data['tipe_diskon'] = data['tipe_diskon'].str.strip().str.capitalize()

# Status: Huruf pertama kapital
data['status'] = data['status'].str.strip().str.capitalize()

data.head(10)

,promo_code,nama_promo,tipe_diskon,nilai_diskon,tanggal_mulai,tanggal_selesai,status
0,KAHFBARU,Diskon Pengguna Baru,Fixed,15000.0,2024-01-01,2024-12-31,Active
1,GAJIAN50,Diskon Gajian,Fixed,20000.0,2024-04-25,2024-05-05,Active
2,RAMADHAN,Promo Ramadhan,Percentage,15.0,2024-03-10,2024-04-10,Expired
3,FACECARE10,Diskon Kategori Face Care,Percentage,10.0,2024-05-01,2024-07-31,Active
4,FLASH25,Flash Sale 25%,Percentage,25.0,2024-06-01,2024-06-03,Expired
5,PARFUMKAHF,Diskon Fragrance,Percentage,12.0,2024-05-01,2024-08-31,Active
6,GRATISONGKIR,Gratis Ongkir,Fixed,10000.0,2024-02-01,2024-02-29,Expired
7,VIP20,Diskon Member Vip,Percentage,20.0,2024-01-15,2024-12-31,Active
8,WEEKEND15,Diskon Akhir Pekan,Percentage,NaN,2024-06-01,2024-08-31,Active
9,CUCI GUDANG,Cuci Gudang Juli,Fixed,15000.0,2024-07-01,2024-07-15,Expired


In [36]:
# Mengisi Missing Value pada kolom nilai diskon dengan cara
# mecari mean dari diskon pada kolom tipe_diskon yang pesentage.
mean_diskon = data.groupby('tipe_diskon')['nilai_diskon'].mean()

print(mean_diskon)

tipe_diskon
Fixed         15000.0
Percentage       16.4
Name: nilai_diskon, dtype: float64


In [37]:
# Hitung rata-rata per tipe diskon
mean_diskon = data.groupby('tipe_diskon')['nilai_diskon'].transform('mean')
# Isi missing value
data['nilai_diskon'] = data['nilai_diskon'].fillna(mean_diskon)
print(data[['tipe_diskon', 'nilai_diskon']])

  tipe_diskon  nilai_diskon
0       Fixed       15000.0
1       Fixed       20000.0
2  Percentage          15.0
3  Percentage          10.0
4  Percentage          25.0
5  Percentage          12.0
6       Fixed       10000.0
7  Percentage          20.0
8  Percentage          16.4
9       Fixed       15000.0


In [38]:
# Mengubah nilai diskon menjadi nilai angka normal
print(data[['nilai_diskon']].isna().sum())
data['nilai_diskon'] = data['nilai_diskon'].astype(int)
data.head(10)

nilai_diskon    0
dtype: int64


,promo_code,nama_promo,tipe_diskon,nilai_diskon,tanggal_mulai,tanggal_selesai,status
0,KAHFBARU,Diskon Pengguna Baru,Fixed,15000,2024-01-01,2024-12-31,Active
1,GAJIAN50,Diskon Gajian,Fixed,20000,2024-04-25,2024-05-05,Active
2,RAMADHAN,Promo Ramadhan,Percentage,15,2024-03-10,2024-04-10,Expired
3,FACECARE10,Diskon Kategori Face Care,Percentage,10,2024-05-01,2024-07-31,Active
4,FLASH25,Flash Sale 25%,Percentage,25,2024-06-01,2024-06-03,Expired
5,PARFUMKAHF,Diskon Fragrance,Percentage,12,2024-05-01,2024-08-31,Active
6,GRATISONGKIR,Gratis Ongkir,Fixed,10000,2024-02-01,2024-02-29,Expired
7,VIP20,Diskon Member Vip,Percentage,20,2024-01-15,2024-12-31,Active
8,WEEKEND15,Diskon Akhir Pekan,Percentage,16,2024-06-01,2024-08-31,Active
9,CUCI GUDANG,Cuci Gudang Juli,Fixed,15000,2024-07-01,2024-07-15,Expired


In [39]:
print("\nMissing values:")
print(data.isnull().sum())


Missing values:
promo_code         0
nama_promo         0
tipe_diskon        0
nilai_diskon       0
tanggal_mulai      0
tanggal_selesai    0
status             0
dtype: int64


In [40]:
# Simpan ke CSV data setelah proses atau clean
data.to_csv(
    "../data/warehouse/promo_clean.csv",
    index=False,
    encoding="utf-8")
# Melihat kembali data yang disimpan
data = pd.read_csv("../data/warehouse/promo_clean.csv")
data.head()

,promo_code,nama_promo,tipe_diskon,nilai_diskon,tanggal_mulai,tanggal_selesai,status
0,KAHFBARU,Diskon Pengguna Baru,Fixed,15000,2024-01-01,2024-12-31,Active
1,GAJIAN50,Diskon Gajian,Fixed,20000,2024-04-25,2024-05-05,Active
2,RAMADHAN,Promo Ramadhan,Percentage,15,2024-03-10,2024-04-10,Expired
3,FACECARE10,Diskon Kategori Face Care,Percentage,10,2024-05-01,2024-07-31,Active
4,FLASH25,Flash Sale 25%,Percentage,25,2024-06-01,2024-06-03,Expired
